In [1]:
!wget -nc https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.2.0/delta-spark_2.12-3.2.0.jar
!wget -nc https://repo1.maven.org/maven2/io/delta/delta-storage/3.2.0/delta-storage-3.2.0.jar


--2026-05-02 18:43:00--  https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.2.0/delta-spark_2.12-3.2.0.jar
Resolving repo1.maven.org (repo1.maven.org)... 104.18.19.12, 104.18.18.12, 2606:4700::6812:120c, ...
Connecting to repo1.maven.org (repo1.maven.org)|104.18.19.12|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6111817 (5.8M) [application/java-archive]
Saving to: ‘delta-spark_2.12-3.2.0.jar’

delta-spark_2.12-3. 100%[===================>]   5.83M  17.3MB/s    in 0.3s    

2026-05-02 18:43:00 (17.3 MB/s) - ‘delta-spark_2.12-3.2.0.jar’ saved [6111817/6111817]

--2026-05-02 18:43:00--  https://repo1.maven.org/maven2/io/delta/delta-storage/3.2.0/delta-storage-3.2.0.jar
Resolving repo1.maven.org (repo1.maven.org)... 104.18.19.12, 2606:4700::6812:120c, 2606:4700::6812:130c
Connecting to repo1.maven.org (repo1.maven.org)|104.18.19.12|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 24946 (24K) [application/java-archive]
Saving to:

In [3]:
import os
from pyspark.sql import SparkSession, functions as F

all_jars = ",".join([
    f"{os.getcwd()}/postgresql-42.7.0.jar",
    f"{os.getcwd()}/delta-spark_2.12-3.2.0.jar",
    f"{os.getcwd()}/delta-storage-3.2.0.jar",
])
spark = (SparkSession.builder
         .appName("Gold-PreCheck")
         .config("spark.jars", all_jars)            # ← ทุก jar ใน classpath
         .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
         .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
         .config("spark.driver.memory", "4g")
         .getOrCreate())

# Verify Delta loaded
loaded = spark.sparkContext._jsc.sc().listJars().toString()
assert "delta-spark" in loaded, f"❌ Delta NOT loaded. Got: {loaded}"
print("✅ Delta jar loaded successfully")


✅ Delta jar loaded successfully


In [4]:
SILVER = "/home/jovyan/work/data/silver"
fact = spark.read.format("delta").load(f"{SILVER}/fact_order_items")
print(f"Silver fact rows: {fact.count():,}")

orphan = fact.filter(F.col("department").isNull()).count()
print(f"Orphan rows (NULL department): {orphan:,}")
print(f"Orphan %: {orphan / fact.count() * 100:.4f}%")

Silver fact rows: 32,434,489
Orphan rows (NULL department): 3
Orphan %: 0.0000%
